# Hari 10-11 — Feature Engineering: Lag, Rolling, dan Encoding

Lanjutan dari `dataset_bersih_minggu2.csv` yang kamu simpan di Hari 9 (sudah melalui koreksi outlier).

Dua bagian: **Hari 10** — fitur lag & rolling average (fondasi model forecasting), **Hari 11** — fitur kalender + encoding, lalu menyusun X/y final siap untuk Modeling di Minggu 3.

---
# BAGIAN A — Hari 10: Fitur Lag & Rolling Average

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("dataset_bersih_minggu2.csv", index_col=0, parse_dates=True)
print(f"Data dimuat: {df.shape[0]} minggu, kolom: {list(df.columns)}")
df.tail()

Data dimuat: 147 minggu, kolom: ['kasus_baru_mingguan', 'vaksin_persen']


,kasus_baru_mingguan,vaksin_persen
Date,,
2022-11-27 00:00:00+00:00,41877.0,76.020068
2022-12-04 00:00:00+00:00,29959.0,75.398628
2022-12-11 00:00:00+00:00,18587.0,75.434557
2022-12-18 00:00:00+00:00,10807.0,75.464296
2022-12-25 00:00:00+00:00,6527.0,75.489253


## Konsep: Fitur Lag dan Data Leakage

**Fitur lag** artinya kita pakai nilai masa lalu sebagai kolom prediktor. Misal `lag_1` = kasus minggu sebelumnya, `lag_2` = kasus dua minggu sebelumnya, dst. Ini masuk akal secara bisnis: untuk memprediksi kasus minggu depan, informasi paling relevan biasanya adalah beberapa minggu terakhir.

**Aturan besi data leakage untuk time-series:** fitur pada baris minggu **t** HANYA boleh berisi informasi dari minggu **t atau sebelumnya**, tidak boleh ada informasi dari **t+1 dst (masa depan)**. Ini beda dengan rolling mean yang kamu pakai di Hari 9 (`center=True`) — itu sengaja memakai data sebelum DAN sesudah, tapi tujuannya untuk **membersihkan catatan historis**, bukan untuk jadi fitur prediksi. Untuk fitur prediksi, rolling window harus **trailing** (hanya mundur ke belakang), tidak boleh `center=True`.

In [2]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# Buat 3 kolom fitur lag dari kolom "kasus_baru_mingguan":
# 1. df["lag_1"] = nilai kasus_baru_mingguan digeser 1 minggu ke belakang (.shift(1))
# 2. df["lag_2"] = digeser 2 minggu (.shift(2))
# 3. df["lag_3"] = digeser 3 minggu (.shift(3))
# Ingat: .shift(1) artinya nilai di baris ini diambil dari baris SEBELUMNYA
# jadi baris paling awal akan otomatis jadi NaN karena tidak ada baris "sebelum" data pertama.
# Setelah selesai, cetak df[["kasus_baru_mingguan", "lag_1", "lag_2", "lag_3"]].head(6) untuk memastikan pola pergeserannya benar.
# Tulis kode kamu di bawah ini:
df["lag_1"] = df["kasus_baru_mingguan"].shift(1)
df["lag_2"] = df["kasus_baru_mingguan"].shift(2)
df["lag_3"] = df["kasus_baru_mingguan"].shift(3)
df[["kasus_baru_mingguan", "lag_1", "lag_2", "lag_3"]].head(6)

,kasus_baru_mingguan,lag_1,lag_2,lag_3
Date,,,,
2020-03-08 00:00:00+00:00,6.0,NaN,NaN,NaN
2020-03-15 00:00:00+00:00,111.0,6.0,NaN,NaN
2020-03-22 00:00:00+00:00,397.0,111.0,6.0,NaN
2020-03-29 00:00:00+00:00,771.0,397.0,111.0,6.0
2020-04-05 00:00:00+00:00,988.0,771.0,397.0,111.0
2020-04-12 00:00:00+00:00,1968.0,988.0,771.0,397.0


In [3]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# Buat fitur rolling average TRAILING (bukan center=True!) dari kasus_baru_mingguan:
# 1. df["rolling_mean_4w"] = rolling mean 4 minggu (.rolling(4).mean()) TANPA center=True
#    (default rolling() di pandas sudah trailing, jadi tidak perlu parameter tambahan)
# 2. Cetak df[["kasus_baru_mingguan", "rolling_mean_4w"]].head(6) untuk mengecek —
#    baris ke-4 (index 3, hitungan dari 0) seharusnya baru punya nilai rolling pertama
#    karena butuh minimal 4 titik data
# Tulis kode kamu di bawah ini:
df["rolling_mean_4w"] = df["kasus_baru_mingguan"].rolling(4).mean()
df[["kasus_baru_mingguan", "rolling_mean_4w"]].head(6)

,kasus_baru_mingguan,rolling_mean_4w
Date,,
2020-03-08 00:00:00+00:00,6.0,NaN
2020-03-15 00:00:00+00:00,111.0,NaN
2020-03-22 00:00:00+00:00,397.0,NaN
2020-03-29 00:00:00+00:00,771.0,321.25
2020-04-05 00:00:00+00:00,988.0,566.75
2020-04-12 00:00:00+00:00,1968.0,1031.00


## Menangani Baris Awal yang Kosong (NaN)

Beberapa baris pertama pasti punya NaN di kolom lag/rolling karena belum cukup histori. Untuk data time-series, baris ini **tidak bisa diisi (imputed)** seperti missing value biasa — solusinya dibuang saja, karena memang belum ada cukup informasi historis di titik itu.

In [4]:
# Cell ini sudah lengkap.
print(f"Shape sebelum buang baris NaN awal: {df.shape}")
df_fitur = df.dropna(subset=["lag_1", "lag_2", "lag_3", "rolling_mean_4w"]).copy()
print(f"Shape sesudah: {df_fitur.shape}")
df_fitur.head()

Shape sebelum buang baris NaN awal: (147, 6)
Shape sesudah: (144, 6)


,kasus_baru_mingguan,vaksin_persen,lag_1,lag_2,lag_3,rolling_mean_4w
Date,,,,,,
2020-03-29 00:00:00+00:00,771.0,0.0,397.0,111.0,6.0,321.25
2020-04-05 00:00:00+00:00,988.0,0.0,771.0,397.0,111.0,566.75
2020-04-12 00:00:00+00:00,1968.0,0.0,988.0,771.0,397.0,1031.00
2020-04-19 00:00:00+00:00,2334.0,0.0,1968.0,988.0,771.0,1515.25
2020-04-26 00:00:00+00:00,2307.0,0.0,2334.0,1968.0,988.0,1899.25


---
# BAGIAN B — Hari 11: Fitur Kalender, Encoding, dan Menyusun X/y Final

Proyek ini tidak punya kolom kategorikal khas seperti "provinsi" atau "jenis kelamin". Tapi dari dekomposisi time-series di Hari 6, kita tahu ada **pola musiman**. Kita bisa tangkap pola itu lewat fitur kalender (bulan), lalu praktik teknik encoding (one-hot) yang biasanya dipakai untuk data kategorikal.

In [5]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Buat kolom baru "bulan" dari index tanggal df_fitur, ambil nomor bulannya
#    (df_fitur.index.month mengembalikan angka 1-12)
# 2. Lakukan one-hot encoding pada kolom "bulan" dengan pd.get_dummies(),
#    beri prefix="bulan" supaya nama kolomnya jelas (jadi bulan_1, bulan_2, dst)
# 3. Gabungkan hasil one-hot encoding itu ke df_fitur dengan pd.concat(axis=1)
# 4. Cetak df_fitur.columns untuk melihat kolom-kolom baru yang muncul
# Tulis kode kamu di bawah ini:
df_fitur["bulan"] = df_fitur.index.month
bulan_encoded = pd.get_dummies(df_fitur["bulan"], prefix="bulan", dtype=int)
df_fitur = pd.concat([df_fitur, bulan_encoded], axis = 1)
df_fitur.columns

Index(['kasus_baru_mingguan', 'vaksin_persen', 'lag_1', 'lag_2', 'lag_3',
       'rolling_mean_4w', 'bulan', 'bulan_1', 'bulan_2', 'bulan_3', 'bulan_4',
       'bulan_5', 'bulan_6', 'bulan_7', 'bulan_8', 'bulan_9', 'bulan_10',
       'bulan_11', 'bulan_12'],
      dtype='object')

## Mendefinisikan Target (y): Kasus Minggu Depan

Ini langkah yang sering terlewat pemula: mengubah data time-series jadi format supervised learning butuh **target eksplisit**. Target kita adalah kasus_baru_mingguan **satu minggu ke depan** relatif terhadap baris saat ini — kebalikan dari fitur lag (yang mundur ke belakang, target ini maju ke depan).

In [6]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Buat kolom "target_minggu_depan" = kasus_baru_mingguan digeser -1
#    (df_fitur["kasus_baru_mingguan"].shift(-1)) — shift NEGATIF artinya menarik nilai
#    dari baris SESUDAHNYA, jadi baris paling akhir otomatis NaN (karena belum ada
#    data minggu depan yang sungguhan)
# 2. Buang baris yang punya NaN di kolom target_minggu_depan dengan .dropna()
#    (assign hasilnya ke df_fitur lagi)
# 3. Cetak df_fitur[["kasus_baru_mingguan", "target_minggu_depan"]].tail(6) untuk
#    memastikan nilai target di tiap baris memang sama dengan kasus_baru_mingguan baris SESUDAHNYA
# Tulis kode kamu di bawah ini:
df_fitur["target_minggu_depan"] = df_fitur["kasus_baru_mingguan"].shift(-1)
df_fitur = df_fitur.dropna(subset=["target_minggu_depan"])
df_fitur[["kasus_baru_mingguan", "target_minggu_depan"]].tail(6)

,kasus_baru_mingguan,target_minggu_depan
Date,,
2022-11-13 00:00:00+00:00,40212.0,46863.0
2022-11-20 00:00:00+00:00,46863.0,41877.0
2022-11-27 00:00:00+00:00,41877.0,29959.0
2022-12-04 00:00:00+00:00,29959.0,18587.0
2022-12-11 00:00:00+00:00,18587.0,10807.0
2022-12-18 00:00:00+00:00,10807.0,6527.0


## Menyusun X (Fitur) dan y (Target) Final

In [7]:
# LATIHAN — tulis kodenya sendiri sesuai panduan komentar di bawah:
# 1. Buat list nama kolom fitur yang dipakai untuk X:
#    ["lag_1", "lag_2", "lag_3", "rolling_mean_4w", "vaksin_persen"] ditambah
#    semua kolom yang namanya diawali "bulan_" (hasil one-hot encoding Hari 11)
#    Hint: [col for col in df_fitur.columns if col.startswith("bulan_")]
# 2. X = df_fitur[kolom_fitur]
# 3. y = df_fitur["target_minggu_depan"]
# 4. Cetak X.shape dan y.shape — jumlah barisnya HARUS sama
# 5. Cetak X.head() untuk melihat hasil akhirnya
# Tulis kode kamu di bawah ini:
kolom_fitur = ["lag_1", "lag_2", "lag_3", "rolling_mean_4w", "vaksin_persen",]

kolom_fitur += [
    col for col in df_fitur.columns
    if col.startswith("bulan_")
]

X = df_fitur[kolom_fitur]
y = df_fitur["target_minggu_depan"]

print("Shape X:", X.shape)
print("Shape y:", y.shape)
print(X.head())

Shape X: (143, 17)
Shape y: (143,)
                            lag_1   lag_2  lag_3  rolling_mean_4w  \
Date                                                                
2020-03-29 00:00:00+00:00   397.0   111.0    6.0           321.25   
2020-04-05 00:00:00+00:00   771.0   397.0  111.0           566.75   
2020-04-12 00:00:00+00:00   988.0   771.0  397.0          1031.00   
2020-04-19 00:00:00+00:00  1968.0   988.0  771.0          1515.25   
2020-04-26 00:00:00+00:00  2334.0  1968.0  988.0          1899.25   

                           vaksin_persen  bulan_1  bulan_2  bulan_3  bulan_4  \
Date                                                                           
2020-03-29 00:00:00+00:00            0.0        0        0        1        0   
2020-04-05 00:00:00+00:00            0.0        0        0        0        1   
2020-04-12 00:00:00+00:00            0.0        0        0        0        1   
2020-04-19 00:00:00+00:00            0.0        0        0        0        1   
2

## Simpan Dataset Siap Modeling

In [8]:
# Cell ini sudah lengkap.
final = pd.concat([X, y], axis=1)
final.to_csv("dataset_siap_modeling.csv")
print("Tersimpan sebagai dataset_siap_modeling.csv")
print(f"Shape akhir: {final.shape}")
final.tail()

Tersimpan sebagai dataset_siap_modeling.csv
Shape akhir: (143, 18)


,lag_1,lag_2,lag_3,rolling_mean_4w,vaksin_persen,bulan_1,bulan_2,bulan_3,bulan_4,bulan_5,bulan_6,bulan_7,bulan_8,bulan_9,bulan_10,bulan_11,bulan_12,target_minggu_depan
Date,,,,,,,,,,,,,,,,,,
2022-11-20 00:00:00+00:00,40212.0,30670.0,19661.0,34351.50,75.984620,0,0,0,0,0,0,0,0,0,0,1,0,41877.0
2022-11-27 00:00:00+00:00,46863.0,40212.0,30670.0,39905.50,76.020068,0,0,0,0,0,0,0,0,0,0,1,0,29959.0
2022-12-04 00:00:00+00:00,41877.0,46863.0,40212.0,39727.75,75.398628,0,0,0,0,0,0,0,0,0,0,0,1,18587.0
2022-12-11 00:00:00+00:00,29959.0,41877.0,46863.0,34321.50,75.434557,0,0,0,0,0,0,0,0,0,0,0,1,10807.0
2022-12-18 00:00:00+00:00,18587.0,29959.0,41877.0,25307.50,75.464296,0,0,0,0,0,0,0,0,0,0,0,1,6527.0


## Refleksi Hari 10-11

> 1. Berapa total baris yang tersisa di dataset final, setelah dikurangi baris-baris NaN dari lag features (di awal) dan target (di akhir)? → 143
> 2. Fitur mana yang menurutmu paling mungkin punya pengaruh besar ke prediksi — lag_1 (minggu lalu) atau vaksin_persen? Kenapa? → lag_1 (minggu lalu) karena korelasi dengan minggu terdekat biasanya paling kuat

---
### Selanjutnya: Hari 12-14 — Scaling, Train-Test Split, dan Mini-Project Minggu 2

- **Hari 12**: Cek apakah fitur perlu di-scale (lag & rolling dalam satuan yang sama dengan target, jadi mungkin tidak wajib untuk model tree-based, tapi tetap wajib untuk model linear)
- **Hari 13**: Train-test split KHUSUS time-series — **tidak boleh acak**. Split harus berurutan waktu: data lama untuk training, data terbaru untuk testing (mensimulasikan kondisi nyata: model hanya "tahu" masa lalu saat prediksi)
- **Hari 14**: Rangkuman pipeline Data Preparation lengkap (Hari 8-14) sebagai mini-project penutup Minggu 2, siap dipakai langsung di Modeling Minggu 3